<style>
.jp-Notebook { max-width: 1000px; margin: 0 auto; }
.jp-RenderedMarkdown h1 { border-bottom: 2px solid #ddd; padding-bottom: .3em; }
.jp-RenderedMarkdown h2 { margin-top: 1.8em; color: #123; border-bottom: 1px solid #eee; }
.jp-RenderedMarkdown h3 { margin-top: 1.4em; color: #345; }
.jp-RenderedHTMLCommon table { margin: 1em 0; }
</style>

# Atividade 3 — MLP: o que uma camada oculta compra

**Redes Neurais — Insper, 2026.2**

O fio condutor da atividade é o que uma camada oculta acrescenta. O Exercício 1 faz um passo de
retropropagação na mão, para ver cada parâmetro se mover. O Exercício 2 usa um dataset que
nenhuma reta resolve, o Exercício 3 passa para três classes com a **mesma** rede e o Exercício 4
pergunta se uma segunda camada oculta ajuda.

**Conformidade com as regras técnicas.** Semente fixa via `rng = np.random.default_rng(42)`, com
**um único objeto `rng` reutilizado em todo o relatório**. O notebook é executado de cima para
baixo (`jupyter nbconvert --execute`), então todos os números desta página são reprodutíveis. O
`rng` é usado duas vezes, nesta ordem: na inicialização dos pesos do Exercício 2 e no sorteio das
sementes dos Exercícios 3 e 4 (cada uma alimenta um gerador próprio, para que as redes rasa e
profunda possam partir da *mesma* semente). Os geradores de dados (`make_blobs`,
`make_classification`) e o `train_test_split` recebem `random_state=42`, como pede o enunciado,
porque não aceitam um `np.random.Generator`.

**Bibliotecas.** NumPy, pandas e Matplotlib. O scikit-learn entra só onde o enunciado permite:
`make_blobs`, `make_classification`, `train_test_split`, `confusion_matrix` e a
`LogisticRegression` das linhas de base. **O MLP — ativações, perdas, passagem direta,
gradientes e atualização — foi escrito à mão em NumPy.** Não há TensorFlow, PyTorch nem
`MLPClassifier`.

**Uso de ferramentas de IA.** Este relatório foi produzido com apoio do Claude (Anthropic) na
escrita do código, na geração das figuras e na redação das análises. As decisões metodológicas,
os parâmetros e as interpretações foram revisados e validados por mim, e consigo explicar cada
trecho do código e cada conclusão apresentada. Todo número pedido pelo enunciado está escrito no
texto, e não apenas na saída das células.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# scikit-learn: só geração/divisão dos dados, matriz de confusão e linhas de base
from sklearn.datasets import make_blobs, make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression

%matplotlib inline

plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (7, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

# ÚNICO gerador aleatório do relatório: criado aqui e reutilizado abaixo,
# na ordem em que as células aparecem.
rng = np.random.default_rng(42)

CORES = ["#1f77b4", "#d62728", "#2ca02c"]

# Exercício 1 — Um passo de retropropagação, na mão

Rede $2 \to 2 \to 1$, $\tanh$ nas duas camadas e perda $L = (y - \hat{y})^2$ ($N = 1$). Valores
dados:

| Símbolo | Valor |
|:---|:---|
| entrada e alvo | $\mathbf{x} = [0{,}5,\ -0{,}2]$, $\ y = 1{,}0$ |
| pesos da camada oculta | $\mathbf{W}^{(1)} = \begin{bmatrix} 0{,}3 & -0{,}1 \\ 0{,}2 & 0{,}4 \end{bmatrix}$ (linha $i$ = pesos do neurônio oculto $i$) |
| biases da camada oculta | $\mathbf{b}^{(1)} = [0{,}1,\ -0{,}2]$ |
| pesos da camada de saída | $\mathbf{W}^{(2)} = [0{,}5,\ -0{,}3]$ |
| bias da camada de saída | $b^{(2)} = 0{,}2$ |
| taxa de aprendizado | $\eta = 0{,}3$ |

Todos os números abaixo foram feitos à mão e estão com 6 casas decimais (o enunciado pede pelo
menos 4). Os valores de $\tanh$ foram arredondados na 6ª casa e os passos seguintes usam os
valores arredondados; a diferença para o cálculo em precisão total aparece só na 7ª casa. Uma
célula de conferência em NumPy, no fim do exercício, refaz as contas — ela não é a resolução,
só a verificação.

## A — Passagem direta

**Pré-ativações da camada oculta**, $\mathbf{z}^{(1)} = \mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$:

$$
\begin{aligned}
z^{(1)}_1 &= 0{,}3 \cdot 0{,}5 + (-0{,}1)(-0{,}2) + 0{,}1 = 0{,}15 + 0{,}02 + 0{,}1 = \mathbf{0{,}270000} \\
z^{(1)}_2 &= 0{,}2 \cdot 0{,}5 + 0{,}4\,(-0{,}2) - 0{,}2 = 0{,}10 - 0{,}08 - 0{,}2 = \mathbf{-0{,}180000}
\end{aligned}
$$

**Ativações da camada oculta**, $\mathbf{h}^{(1)} = \tanh(\mathbf{z}^{(1)})$:

$$
h^{(1)}_1 = \tanh(0{,}27) = \mathbf{0{,}263625}, \qquad
h^{(1)}_2 = \tanh(-0{,}18) = \mathbf{-0{,}178081}
$$

**Pré-ativação de saída**, $u^{(2)} = \mathbf{W}^{(2)}\mathbf{h}^{(1)} + b^{(2)}$:

$$
u^{(2)} = 0{,}5 \cdot 0{,}263625 + (-0{,}3)(-0{,}178081) + 0{,}2
        = 0{,}131812 + 0{,}053424 + 0{,}2 = \mathbf{0{,}385237}
$$

**Saída e perda:**

$$
\hat{y} = \tanh(0{,}385237) = \mathbf{0{,}367247}, \qquad
L = (1 - 0{,}367247)^2 = 0{,}632753^2 = \mathbf{0{,}400377}
$$

## B — Passagem reversa

A regra da cadeia é aplicada de fora para dentro. A cada camada, o gradiente que chega
($\partial L/\partial \text{saída}$) é multiplicado pela derivada local.

**1. Derivada da perda em relação à saída:**

$$
\frac{\partial L}{\partial \hat{y}} = -2\,(y - \hat{y}) = -2 \cdot 0{,}632753 = \mathbf{-1{,}265507}
$$

**2. Através do $\tanh$ de saída.** Como $\frac{d}{du}\tanh(u) = 1 - \tanh^2(u)$ e
$\tanh(u^{(2)}) = \hat{y}$, a derivada local já está disponível sem recalcular nada:
$1 - \hat{y}^2 = 1 - 0{,}367247^2 = 1 - 0{,}134870 = 0{,}865130$.

$$
\delta^{(2)} \equiv \frac{\partial L}{\partial u^{(2)}}
  = \frac{\partial L}{\partial \hat{y}}\,(1 - \hat{y}^2)
  = -1{,}265507 \cdot 0{,}865130 = \mathbf{-1{,}094828}
$$

**3. Camada de saída.** Como $u^{(2)} = \sum_j W^{(2)}_j h^{(1)}_j + b^{(2)}$, temos
$\partial u^{(2)}/\partial W^{(2)}_j = h^{(1)}_j$ e $\partial u^{(2)}/\partial b^{(2)} = 1$:

$$
\begin{aligned}
\frac{\partial L}{\partial \mathbf{W}^{(2)}} &= \delta^{(2)}\,\mathbf{h}^{(1)}
  = -1{,}094828 \cdot [0{,}263625,\ -0{,}178081] = \mathbf{[-0{,}288624,\ \ 0{,}194968]} \\
\frac{\partial L}{\partial b^{(2)}} &= \delta^{(2)} = \mathbf{-1{,}094828}
\end{aligned}
$$

**4. Propagando de volta para a camada oculta.** Como $\partial u^{(2)}/\partial h^{(1)}_j = W^{(2)}_j$:

$$
\frac{\partial L}{\partial \mathbf{h}^{(1)}} = \delta^{(2)}\,\mathbf{W}^{(2)}
  = -1{,}094828 \cdot [0{,}5,\ -0{,}3] = \mathbf{[-0{,}547414,\ \ 0{,}328448]}
$$

Através do $\tanh$ oculto, com $1 - (h^{(1)}_1)^2 = 1 - 0{,}069498 = 0{,}930502$ e
$1 - (h^{(1)}_2)^2 = 1 - 0{,}031713 = 0{,}968287$ (produto elemento a elemento):

$$
\boldsymbol{\delta}^{(1)} \equiv \frac{\partial L}{\partial \mathbf{z}^{(1)}}
  = \frac{\partial L}{\partial \mathbf{h}^{(1)}} \odot \big(1 - (\mathbf{h}^{(1)})^2\big)
  = [-0{,}547414 \cdot 0{,}930502,\ \ 0{,}328448 \cdot 0{,}968287]
  = \mathbf{[-0{,}509370,\ \ 0{,}318032]}
$$

**5. Camada oculta.** Como $z^{(1)}_i = \sum_k W^{(1)}_{ik} x_k + b^{(1)}_i$, temos
$\partial L/\partial W^{(1)}_{ik} = \delta^{(1)}_i\,x_k$ — o produto externo
$\boldsymbol{\delta}^{(1)}\mathbf{x}^\top$:

$$
\frac{\partial L}{\partial \mathbf{W}^{(1)}}
 = \begin{bmatrix} -0{,}509370 \cdot 0{,}5 & -0{,}509370 \cdot (-0{,}2) \\
                    0{,}318032 \cdot 0{,}5 &  0{,}318032 \cdot (-0{,}2) \end{bmatrix}
 = \begin{bmatrix} \mathbf{-0{,}254685} & \mathbf{\ \ 0{,}101874} \\
                   \mathbf{\ \ 0{,}159016} & \mathbf{-0{,}063606} \end{bmatrix},
\qquad
\frac{\partial L}{\partial \mathbf{b}^{(1)}} = \boldsymbol{\delta}^{(1)} = \mathbf{[-0{,}509370,\ \ 0{,}318032]}
$$

## C — Atualização dos parâmetros

Cada parâmetro anda $-\eta \cdot$ gradiente, com $\eta = 0{,}3$:

| Parâmetro | Antes | Gradiente | $-\eta \cdot$ gradiente | **Depois** |
|:---|---:|---:|---:|---:|
| $W^{(2)}_1$ | $0{,}5$ | $-0{,}288624$ | $+0{,}086587$ | $\mathbf{0{,}586587}$ |
| $W^{(2)}_2$ | $-0{,}3$ | $0{,}194968$ | $-0{,}058490$ | $\mathbf{-0{,}358490}$ |
| $b^{(2)}$ | $0{,}2$ | $-1{,}094828$ | $+0{,}328448$ | $\mathbf{0{,}528448}$ |
| $W^{(1)}_{11}$ | $0{,}3$ | $-0{,}254685$ | $+0{,}076405$ | $\mathbf{0{,}376405}$ |
| $W^{(1)}_{12}$ | $-0{,}1$ | $0{,}101874$ | $-0{,}030562$ | $\mathbf{-0{,}130562}$ |
| $W^{(1)}_{21}$ | $0{,}2$ | $0{,}159016$ | $-0{,}047705$ | $\mathbf{0{,}152295}$ |
| $W^{(1)}_{22}$ | $0{,}4$ | $-0{,}063606$ | $+0{,}019082$ | $\mathbf{0{,}419082}$ |
| $b^{(1)}_1$ | $0{,}1$ | $-0{,}509370$ | $+0{,}152811$ | $\mathbf{0{,}252811}$ |
| $b^{(1)}_2$ | $-0{,}2$ | $0{,}318032$ | $-0{,}095410$ | $\mathbf{-0{,}295410}$ |

Ou seja,
$\mathbf{W}^{(2)} = [0{,}586587,\ -0{,}358490]$, $b^{(2)} = 0{,}528448$,
$\mathbf{W}^{(1)} = \begin{bmatrix} 0{,}376405 & -0{,}130562 \\ 0{,}152295 & 0{,}419082 \end{bmatrix}$ e
$\mathbf{b}^{(1)} = [0{,}252811,\ -0{,}295410]$. (São nove números porque $\mathbf{W}^{(1)}$ tem
quatro entradas; o enunciado conta os "oito parâmetros" por grupo.)

**A perda numa segunda passagem direta desceria.** O motivo: o gradiente aponta a direção de
maior *aumento* local de $L$, e cada parâmetro andou no sentido oposto. Em primeira ordem,
$\Delta L \approx \nabla L \cdot \Delta\boldsymbol\theta = -\eta\,\lVert\nabla L\rVert^2 < 0$.
Dá para ver a lógica parâmetro a parâmetro: $\hat{y} = 0{,}367$ está abaixo do alvo $1{,}0$, então
$\delta^{(2)} < 0$ e todas as mudanças empurram $u^{(2)}$ para cima — $b^{(2)}$ sobe; $W^{(2)}_1$
sobe porque multiplica um $h^{(1)}_1 > 0$; $W^{(2)}_2$ desce porque multiplica um
$h^{(1)}_2 < 0$; e a camada oculta muda para tornar $h^{(1)}_1$ maior e $h^{(1)}_2$ mais negativo.
A ressalva é que a garantia é só local: com $\eta$ grande demais, o passo pode passar do mínimo e
a perda subir. Com $\eta = 0{,}3$ isso não acontece — refazendo a passagem direta com os
parâmetros novos, $\hat{y}$ vai de $0{,}3672$ para $0{,}7112$ e $L$ cai de $0{,}4004$ para
$\mathbf{0{,}0834}$ (conferência abaixo).

In [ ]:
# Conferência em NumPy das contas feitas à mão acima (não é a resolução).
x  = np.array([0.5, -0.2]); y_alvo = 1.0
W1 = np.array([[0.3, -0.1], [0.2, 0.4]]); b1 = np.array([0.1, -0.2])
W2 = np.array([0.5, -0.3]);               b2 = 0.2
eta = 0.3

def passagem_direta(W1, b1, W2, b2):
    z1 = W1 @ x + b1; h1 = np.tanh(z1)
    u2 = W2 @ h1 + b2; y_hat = np.tanh(u2)
    return z1, h1, u2, y_hat, (y_alvo - y_hat) ** 2

z1, h1, u2, y_hat, L = passagem_direta(W1, b1, W2, b2)
dL_dyhat = -2 * (y_alvo - y_hat)
d2 = dL_dyhat * (1 - y_hat**2)            # dL/du2
dW2, db2 = d2 * h1, d2
dh1 = d2 * W2
d1 = dh1 * (1 - h1**2)                    # dL/dz1
dW1, db1 = np.outer(d1, x), d1

W1n, b1n, W2n, b2n = W1 - eta*dW1, b1 - eta*db1, W2 - eta*dW2, b2 - eta*db2
*_, y_hat_n, L_n = passagem_direta(W1n, b1n, W2n, b2n)

np.set_printoptions(precision=6, suppress=True)
print("z1 =", z1, "  h1 =", h1)
print(f"u2 = {u2:.6f}   y_hat = {y_hat:.6f}   L = {L:.6f}")
print(f"dL/dy_hat = {dL_dyhat:.6f}   dL/du2 = {d2:.6f}")
print("dL/dW2 =", dW2, f"  dL/db2 = {db2:.6f}")
print("dL/dh1 =", dh1, "  dL/dz1 =", d1)
print("dL/dW1 =\n", dW1, "\ndL/db1 =", db1)
print("\nW2 novo =", W2n, f"  b2 novo = {b2n:.6f}")
print("W1 novo =\n", W1n, "\nb1 novo =", b1n)
print(f"\n2ª passagem: y_hat = {y_hat_n:.6f}   L = {L_n:.6f}   (era {L:.6f})")

# A implementação do MLP (usada nos Exercícios 2, 3 e 4)

Um único conjunto de funções serve aos três exercícios. A rede é descrita por uma lista de
tamanhos de camada (`[2, 8, 1]`, `[4, 16, 3]`, `[4, 16, 16, 3]`…), então o número de camadas é
argumento, não constante. As camadas ocultas usam $\tanh$. A camada de saída e a perda vêm juntas
num objeto **cabeça** (`CABECA_BINARIA` ou `CABECA_MULTICLASSE`), que também é argumento.

**Convenções.** As amostras ficam nas linhas: $\mathbf{X}$ tem forma $(N, d)$. A camada $\ell$
tem $\mathbf{W}^{(\ell)}$ de forma $(n_{\ell-1}, n_\ell)$ — a transposta da convenção do
Exercício 1 — para que a passagem direta de um lote inteiro seja
$\mathbf{U}^{(\ell)} = \mathbf{H}^{(\ell-1)}\mathbf{W}^{(\ell)} + \mathbf{b}^{(\ell)}$, com
$\mathbf{H}^{(0)} = \mathbf{X}$.

**Passagem reversa.** Seja $\boldsymbol\Delta^{(\ell)} = \partial L / \partial \mathbf{U}^{(\ell)}$.
A cabeça entrega $\boldsymbol\Delta^{(L)}$ na saída e, a partir dela, para $\ell = L, \dots, 1$:

$$
\frac{\partial L}{\partial \mathbf{W}^{(\ell)}} = (\mathbf{H}^{(\ell-1)})^\top \boldsymbol\Delta^{(\ell)}, \qquad
\frac{\partial L}{\partial \mathbf{b}^{(\ell)}} = \textstyle\sum_{n} \boldsymbol\Delta^{(\ell)}_{n,:}, \qquad
\boldsymbol\Delta^{(\ell-1)} = \big(\boldsymbol\Delta^{(\ell)} (\mathbf{W}^{(\ell)})^\top\big) \odot \big(1 - (\mathbf{H}^{(\ell-1)})^2\big).
$$

São as mesmas três contas do Exercício 1 (gradiente dos pesos, do bias e propagação pelo
$\tanh$), escritas para um lote e para qualquer profundidade.

**A cabeça binária** é sigmoide + entropia cruzada binária (BCE). Com
$p = \sigma(u)$ e $L = -\frac{1}{N}\sum_n [y_n \log p_n + (1-y_n)\log(1-p_n)]$:
$\frac{\partial L}{\partial p_n} = \frac{1}{N}\frac{p_n - y_n}{p_n(1-p_n)}$ e
$\sigma'(u) = p(1-p)$, logo $\frac{\partial L}{\partial u_n} = \frac{p_n - y_n}{N}$. O fator
$p(1-p)$ se cancela. A cabeça multiclasse (softmax + entropia cruzada categórica) chega ao
**mesmo** $(\mathbf{P} - \mathbf{Y})/N$; a derivação está no item 3.B.

**Inicialização e treino.** Pesos $\sim \mathcal{N}(0,\ 1/n_{\ell-1})$ (escala de LeCun/Xavier,
que mantém a variância das pré-ativações perto de 1 e o $\tanh$ fora da saturação), biases em
zero. Treino por **gradiente descendente em lote completo**: a cada época, uma passagem direta
sobre todo o conjunto de treino, uma reversa e uma atualização. Sem minilotes não há
embaralhamento, e a única aleatoriedade do treino é a inicialização — o que deixa a comparação
do Exercício 4 controlada por construção.

In [ ]:
# ----------------------------------------------------------------------------
# Ativações e cabeças (camada de saída + perda)
# ----------------------------------------------------------------------------
def sigmoide(u):
    """sigma(u) = 1 / (1 + e^-u), escrita de forma estável para u muito negativo."""
    e = np.exp(-np.abs(u))
    return np.where(u >= 0, 1 / (1 + e), e / (1 + e))


def softmax(U):
    """Softmax por linha. Subtrair o máximo da linha não muda o resultado e evita overflow."""
    E = np.exp(U - U.max(axis=1, keepdims=True))
    return E / E.sum(axis=1, keepdims=True)


def bce(P, Y):
    """Entropia cruzada binária média. P, Y: (N, 1), Y em {0, 1}."""
    P = np.clip(P, 1e-12, 1 - 1e-12)
    return float(-np.mean(Y * np.log(P) + (1 - Y) * np.log(1 - P)))


def cce(P, Y):
    """Entropia cruzada categórica média. P: (N, K) probabilidades, Y: (N, K) one-hot."""
    return float(-np.mean(np.sum(Y * np.log(np.clip(P, 1e-12, 1)), axis=1)))


def grad_saida(P, Y):
    """dL/dU na camada de saída: (P - Y) / N.

    Vale para os DOIS pares sigmoide + BCE e softmax + CCE (derivações no texto).
    """
    return (P - Y) / len(Y)


# Uma cabeça = ativação de saída + perda + dL/dU. É o único ponto que muda
# entre a classificação binária e a multiclasse.
CABECA_BINARIA     = {"ativacao": sigmoide, "perda": bce, "grad": grad_saida}
CABECA_MULTICLASSE = {"ativacao": softmax,  "perda": cce, "grad": grad_saida}


# ----------------------------------------------------------------------------
# Núcleo do MLP: inicialização, passagem direta, passagem reversa e atualização.
# Este bloco é usado sem nenhuma alteração nos Exercícios 2, 3 e 4.
# ----------------------------------------------------------------------------
def inicializar(tamanhos, gerador):
    """Lista de (W, b), uma por camada. W ~ N(0, 1/n_entrada), b = 0.

    tamanhos: [n_entrada, n_oculta_1, ..., n_saida], com qualquer profundidade.
    """
    return [(gerador.normal(0.0, np.sqrt(1.0 / n_in), size=(n_in, n_out)), np.zeros(n_out))
            for n_in, n_out in zip(tamanhos[:-1], tamanhos[1:])]


def passagem_direta(params, X, cabeca):
    """Retorna (P, H): as saídas da rede e as ativações de cada camada.

    H[0] = X e H[l] = tanh(H[l-1] W_l + b_l) nas camadas ocultas; a última
    camada aplica a ativação da cabeça (sigmoide ou softmax). H é guardado
    porque a passagem reversa precisa dele.
    """
    H = [X]
    for W, b in params[:-1]:
        H.append(np.tanh(H[-1] @ W + b))
    W, b = params[-1]
    P = cabeca["ativacao"](H[-1] @ W + b)
    return P, H


def passagem_reversa(params, H, delta):
    """Gradientes [(dW, db), ...] de todas as camadas, da saída para a entrada.

    delta: dL/dU da camada de saída, entregue pela cabeça.
    Em cada camada l:  dW = H[l-1]^T delta,  db = soma de delta nas linhas,
    e o delta da camada anterior é (delta W^T) * tanh'(.) = (delta W^T) * (1 - H^2).
    """
    grads = [None] * len(params)
    for l in range(len(params) - 1, -1, -1):
        W, _ = params[l]
        grads[l] = (H[l].T @ delta, delta.sum(axis=0))
        if l > 0:                                   # H[0] é a entrada: não há tanh antes dela
            delta = (delta @ W.T) * (1.0 - H[l] ** 2)
    return grads


def atualizar(params, grads, eta):
    """Um passo de gradiente descendente: theta <- theta - eta * dL/dtheta."""
    return [(W - eta * dW, b - eta * db) for (W, b), (dW, db) in zip(params, grads)]


# ----------------------------------------------------------------------------
# Laço de treino e utilitários
# ----------------------------------------------------------------------------
def treinar_mlp(tamanhos, X, Y, cabeca, eta, epocas, gerador):
    """Gradiente descendente em lote completo. Retorna (params, perda de treino por época).

    A perda registrada na época t é a dos parâmetros ANTES da atualização t.
    """
    params = inicializar(tamanhos, gerador)
    historico = np.empty(epocas)
    for t in range(epocas):
        P, H = passagem_direta(params, X, cabeca)
        historico[t] = cabeca["perda"](P, Y)
        params = atualizar(params, passagem_reversa(params, H, cabeca["grad"](P, Y)), eta)
    return params, historico


def prever(params, X, cabeca):
    """Classe prevista: limiar 0,5 na saída binária, argmax na multiclasse."""
    P, _ = passagem_direta(params, X, cabeca)
    return (P[:, 0] >= 0.5).astype(int) if P.shape[1] == 1 else P.argmax(axis=1)


def contar_parametros(tamanhos):
    return sum(n_in * n_out + n_out for n_in, n_out in zip(tamanhos[:-1], tamanhos[1:]))

**Teste do gradiente.** Antes de treinar qualquer coisa, a passagem reversa é comparada com
diferenças finitas centradas, $\frac{L(\theta + \varepsilon) - L(\theta - \varepsilon)}{2\varepsilon}$,
em todos os parâmetros de uma rede pequena com duas camadas ocultas, nas duas cabeças. Se a
derivação estiver certa, o erro relativo fica na ordem de $10^{-7}$ ou menor. (Os dados e pesos
deste teste vêm de um gerador descartável com semente 0, para não consumir o `rng` do
relatório.)

In [ ]:
def checar_gradiente(tamanhos, cabeca, Y_de, eps=1e-6):
    g = np.random.default_rng(0)                         # gerador descartável, só para o teste
    X = g.normal(size=(7, tamanhos[0]))
    Y = Y_de(g, 7)
    params = inicializar(tamanhos, g)
    P, H = passagem_direta(params, X, cabeca)
    analitico = passagem_reversa(params, H, cabeca["grad"](P, Y))

    pior = 0.0
    for l, (W, b) in enumerate(params):
        for k, A in enumerate((W, b)):
            for idx in np.ndindex(A.shape):
                orig = A[idx]
                A[idx] = orig + eps; Lp = cabeca["perda"](passagem_direta(params, X, cabeca)[0], Y)
                A[idx] = orig - eps; Lm = cabeca["perda"](passagem_direta(params, X, cabeca)[0], Y)
                A[idx] = orig
                num, ana = (Lp - Lm) / (2 * eps), analitico[l][k][idx]
                pior = max(pior, abs(num - ana) / max(1e-8, abs(num) + abs(ana)))
    return pior

e_bin = checar_gradiente([3, 5, 4, 1], CABECA_BINARIA,
                         lambda g, n: g.integers(0, 2, size=(n, 1)).astype(float))
e_mul = checar_gradiente([3, 5, 4, 3], CABECA_MULTICLASSE,
                         lambda g, n: np.eye(3)[g.integers(0, 3, size=n)])
print(f"maior erro relativo, cabeça binária    : {e_bin:.2e}")
print(f"maior erro relativo, cabeça multiclasse: {e_mul:.2e}")

# Exercício 2 — Classificação binária que uma reta não resolve

## A — Gerar os dados

Parâmetros do enunciado, sem alterações: três grupos com $\sigma = 1{,}2$, centrados em
$(0, 0)$ (500 pontos, Classe 0), $(3, 3)$ e $(-3, -3)$ (250 pontos cada, ambos Classe 1),
`random_state=42`. Divisão 80/20 estratificada com `random_state=42`.

In [ ]:
centers = np.array([[0., 0.], [3., 3.], [-3., -3.]])
X2, c2 = make_blobs(n_samples=[500, 250, 250], centers=centers,
                    cluster_std=1.2, random_state=42)
y2 = (c2 > 0).astype(int)          # classe 0: 1 grupo | classe 1: 2 grupos

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2,
                                              random_state=42, stratify=y2)

print(f"total: {X2.shape}   treino: {X2_tr.shape}   teste: {X2_te.shape}")
print("por classe — total:", np.bincount(y2), " treino:", np.bincount(y2_tr),
      " teste:", np.bincount(y2_te))
print("média da Classe 0:", X2[y2 == 0].mean(axis=0).round(3),
      "  média da Classe 1:", X2[y2 == 1].mean(axis=0).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for k, nome in [(0, "Classe 0 (grupo central)"), (1, "Classe 1 (dois grupos)")]:
    ax.scatter(X2[y2 == k, 0], X2[y2 == k, 1], s=10, alpha=0.6, color=CORES[k], label=nome)
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_title("Figura 1 — Exercício 2: os 1000 pontos coloridos por classe")
ax.legend(loc="upper left", markerscale=2)
ax.set_aspect("equal")
plt.show()

A Figura 1 mostra a Classe 0 no meio e a Classe 1 dividida em dois grupos em lados opostos, ao
longo da diagonal $x_1 = x_2$. Um detalhe importa para o item B: a **média** da Classe 1 cai em
cima da média da Classe 0 (as duas ficam perto da origem). As duas classes têm o mesmo centro.

## B — Linha de base linear

In [ ]:
base2 = LogisticRegression().fit(X2_tr, y2_tr)
acc_base2 = base2.score(X2_te, y2_te)
print(f"regressão logística — acurácia no teste: {acc_base2*100:.2f}%")
print(f"coeficientes w = {base2.coef_[0].round(4)},  b = {base2.intercept_[0]:.4f}")
print(f"fração de teste prevista como Classe 1: {base2.predict(X2_te).mean()*100:.1f}%")
print(f"probabilidade prevista no teste: de {base2.predict_proba(X2_te)[:, 1].min():.3f} "
      f"a {base2.predict_proba(X2_te)[:, 1].max():.3f}")

# Para comparação: a melhor reta da família x1 + x2 = c, que isola só o grupo de
# cima da Classe 1. O limiar c é escolhido no TREINO e avaliado no teste.
soma_tr, soma_te = X2_tr.sum(axis=1), X2_te.sum(axis=1)
c_reta = max(np.linspace(-6, 6, 1201), key=lambda c: ((soma_tr > c) == y2_tr).mean())
print(f"\nmelhor reta x1 + x2 = {c_reta:.2f}:  treino {((soma_tr > c_reta) == y2_tr).mean()*100:.2f}%"
      f"   teste {((soma_te > c_reta) == y2_te).mean()*100:.2f}%")

A regressão logística acerta **47,50%** no teste: pior do que chutar. Esse é o número usado no
item D.

## C — Implementar e treinar o MLP

O MLP é o da seção de implementação, com a cabeça binária. **Arquitetura $2 \to 8 \to 1$**
($\tanh$ na camada oculta, sigmoide na saída, BCE), 33 parâmetros, $\eta = 0{,}3$, **2000
épocas** em lote completo. Os pesos iniciais saem do `rng` global (seu primeiro uso).

A escolha foi feita olhando o *conjunto de treino* (um teste rápido com 4, 8 e 16 neurônios e
$\eta \in \{0{,}1;\ 0{,}3;\ 1{,}0\}$ deu acurácias entre 91,5% e 92,5% em todos os casos). Duas
unidades ocultas já bastariam em princípio (item D), mas 8 dão folga para a otimização. Os
dados de entrada não foram padronizados: as duas coordenadas já estão na mesma escala, com
desvio-padrão perto de 3.

In [ ]:
ARQ2, ETA2, EPOCAS2 = [2, 8, 1], 0.3, 2000

params2, perda2 = treinar_mlp(ARQ2, X2_tr, y2_tr[:, None], CABECA_BINARIA,
                              eta=ETA2, epocas=EPOCAS2, gerador=rng)   # 1º uso do rng

acc_tr2 = (prever(params2, X2_tr, CABECA_BINARIA) == y2_tr).mean()
acc_te2 = (prever(params2, X2_te, CABECA_BINARIA) == y2_te).mean()
print(f"arquitetura {ARQ2}  ({contar_parametros(ARQ2)} parâmetros),  eta = {ETA2},  {EPOCAS2} épocas")
print(f"perda de treino: época 1 = {perda2[0]:.4f}   época {EPOCAS2} = {perda2[-1]:.4f}")
print(f"acurácia — treino: {acc_tr2*100:.2f}%   teste: {acc_te2*100:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(np.arange(1, EPOCAS2 + 1), perda2, lw=1.8, color="#9467bd",
        label=f"MLP {ARQ2}, $\\eta = {ETA2}$")
ax.axhline(np.log(2), ls="--", color="gray", lw=1, label=r"$\ln 2$ (prever 0,5 sempre)")
ax.set_xlabel("época"); ax.set_ylabel("perda de treino (BCE)")
ax.set_title("Figura 2 — Exercício 2: perda de treino × época")
ax.legend()
plt.show()

In [ ]:
def plotar_fronteira_binaria(params, X, y, ax, passo=0.05):
    # avalia a rede numa grade e sombreia as regiões P >= 0,5 e P < 0,5
    x1 = np.arange(X[:, 0].min() - 1.5, X[:, 0].max() + 1.5, passo)
    x2 = np.arange(X[:, 1].min() - 1.5, X[:, 1].max() + 1.5, passo)
    G1, G2 = np.meshgrid(x1, x2)
    P = passagem_direta(params, np.c_[G1.ravel(), G2.ravel()], CABECA_BINARIA)[0].reshape(G1.shape)
    ax.contourf(G1, G2, (P >= 0.5).astype(int), levels=[-0.5, 0.5, 1.5],
                cmap=ListedColormap(["#c6dbef", "#fcbba1"]), alpha=0.8)
    ax.contour(G1, G2, P, levels=[0.5], colors="k", linewidths=1.5)
    for k, nome in [(0, "Classe 0"), (1, "Classe 1")]:
        ax.scatter(X[y == k, 0], X[y == k, 1], s=16, color=CORES[k],
                   edgecolors="white", linewidths=0.4, label=f"{nome} (teste)")

fig, ax = plt.subplots(figsize=(6.5, 6))
plotar_fronteira_binaria(params2, X2_te, y2_te, ax)
erros2 = prever(params2, X2_te, CABECA_BINARIA) != y2_te
ax.scatter(X2_te[erros2, 0], X2_te[erros2, 1], s=70, facecolors="none", edgecolors="k",
           lw=1.2, label=f"erros ({erros2.sum()})")
ax.plot([], [], color="k", lw=1.5, label="fronteira $P = 0{,}5$")
ax.fill_between([], [], color="#fcbba1", label="região prevista Classe 1")
ax.fill_between([], [], color="#c6dbef", label="região prevista Classe 0")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
ax.set_title(f"Figura 3 — Exercício 2: fronteira de decisão do MLP {ARQ2} (teste)")
ax.legend(loc="upper left", fontsize=8)
ax.set_aspect("equal")
plt.show()

**Resultados.** Arquitetura $2 \to 8 \to 1$ (33 parâmetros), $\eta = 0{,}3$, 2000 épocas em lote
completo. A perda de treino cai de 0,7930 para **0,1062** (Figura 2) e a acurácia no teste é
**91,50%** (183 de 200), dentro da faixa de 90–93% esperada. No treino ela é 96,50%. Parte da
diferença é ruído do teste: com 200 pontos, cada ponto vale 0,5 p.p., e o erro-padrão de uma
acurácia de ~92% é de ~2 p.p.

A Figura 2 tem três fases. A perda começa acima de $\ln 2 \approx 0{,}693$ (o valor de prever 0,5
para todo mundo), cai até ele quase de imediato — a rede aprende primeiro que as classes são
equilibradas — e encosta nesse patamar por algumas épocas: nesse ponto a rede ainda é
praticamente linear e, como a regressão logística do item B, não tem o que ganhar. Depois
**despenca** entre as épocas ~20 e ~150, quando as unidades ocultas se alinham com a diagonal e a
rede "descobre" a faixa. Daí em diante a descida é lenta: da época 500 à 2000 a perda só vai de
~0,12 para 0,106, e o que muda é o ajuste fino das bordas da faixa.

## D — Análise

In [ ]:
# Como cada unidade oculta enxerga o plano: a reta onde a pré-ativação dela é zero
W_h, b_h = params2[0]
W_o, b_o = params2[1]
unidades = pd.DataFrame({
    "w_x1": W_h[0], "w_x2": W_h[1], "b": b_h, "peso na saída": W_o[:, 0],
    "ângulo da normal (graus)": np.degrees(np.arctan2(W_h[1], W_h[0])),
    "distância da reta à origem": -b_h / np.linalg.norm(W_h, axis=0),
}).round(3)
unidades.index.name = "unidade"
print(f"bias da saída: {b_o[0]:.3f}")
unidades

**Por que a reta fica abaixo de 50%.** Uma reta divide o plano em dois semiplanos e, na
regressão logística, $P(\text{Classe 1})$ só pode crescer numa direção. Nestes dados, a Classe 1
cresce nas **duas** pontas da diagonal, e a Classe 0 está no meio. Pior: as duas classes têm a
*mesma* média, perto da origem (item A). O gradiente da log-verossimilhança da regressão
logística em $\mathbf{w}$ é proporcional a $\sum_n (y_n - p_n)\,\mathbf{x}_n$, que, com $p$ constante,
vale aproximadamente $N_1(\bar{\mathbf{x}}_1 - \bar{\mathbf{x}})$ — e isso é quase zero. O ótimo da
perda logística fica, portanto, perto de $\mathbf{w} \approx \mathbf{0}$: os coeficientes
ajustados são $\mathbf{w} = [0{,}0296,\ 0{,}0015]$ e $b = -0{,}0031$, e a probabilidade prevista
fica entre 0,457 e 0,544 em **todos** os pontos de teste. A "fronteira" que resta, $P = 0{,}5$, é
uma reta quase vertical perto de $x_1 \approx 0{,}1$, definida por flutuações de amostra. Ela passa
pelo meio da nuvem da Classe 0 e pelo vão entre os dois grupos da Classe 1 — põe metade da
Classe 0 e o grupo de cima de um lado, a outra metade e o grupo de baixo do outro. Não carrega
informação nenhuma, então a acurácia fica no nível do acaso; o fato de ser 47,50% e não
exatamente 50% é ruído de amostra (a orientação ajustada no treino não se sustenta no teste).

Não é verdade, porém, que **nenhuma** reta faz melhor que o acaso. A melhor reta da família
$x_1 + x_2 = c$, com $c$ escolhido no treino ($c = 3{,}41$), isola só o grupo de cima e acerta
74,12% no treino e 69,00% no teste, porque classifica corretamente quase toda a Classe 0 e um dos
dois grupos da Classe 1. O que nenhuma reta faz é acertar *os dois* grupos ao mesmo tempo: em
teoria, uma fronteira linear erra pelo menos um grupo inteiro (25% dos dados) e fica num teto de
~72%. A regressão logística nem chega lá, porque a perda que ela minimiza, por simetria, não
prefere nenhum dos dois grupos para sacrificar e acaba sem escolher nenhum.

**O que o MLP faz (Figura 3).** A região que a rede atribui à Classe 1 (em vermelho) **não é
conexa**: são duas regiões separadas, uma no canto superior direito e outra no inferior
esquerdo, com uma **faixa** diagonal da Classe 0 entre elas. A fronteira é formada por duas
curvas quase paralelas, perpendiculares à diagonal $x_1 = x_2$, uma em cada lado da origem. Formalmente,
a região da Classe 0 é a interseção de dois semiplanos, $\{-a < x_1 + x_2 < a\}$, e a da Classe 1
é o complemento dessa faixa — a **união** de dois semiplanos opostos. Uma reta sozinha produz um
único semiplano, sempre conexo e convexo. Não há reta que deixe a Classe 1 dos dois lados.

A tabela das unidades ocultas mostra como a rede constrói isso. Cada unidade $\tanh$ é uma
"reta suave": vale perto de $+1$ de um lado da sua reta e perto de $-1$ do outro. As cinco
unidades com mais peso na saída (3, 6, 7, 1 e 2, com $|$peso$|$ entre 1,8 e 3,1) têm a normal a
menos de ~20° da diagonal — orientadas para $38°$–$58°$ ou para $205°$–$212°$
($-155°$ a $-149°$) —, ou seja, as retas delas são quase perpendiculares a $x_1 = x_2$ e estão a
distâncias entre 1,1 e 2,8 da origem, dos dois lados. As unidades 0 e 5 têm a normal ao longo
da antidiagonal; a 0 tem peso desprezível na saída ($-0{,}31$), e a 5 só entorta a fronteira
nas bordas da faixa. A camada de saída soma essas
"retas" com pesos de sinais escolhidos para que a soma seja alta nas duas pontas e baixa no
meio, algo como $\tanh(x_1 + x_2 - a) - \tanh(x_1 + x_2 + a)$, com sinal trocado. É por isso que
**duas** unidades ocultas já seriam suficientes em princípio: uma faixa precisa de duas retas.
Uma camada oculta transforma o problema num espaço $\mathbf{h}$ onde as classes ficam linearmente
separáveis, e a camada de saída é, ela mesma, uma regressão logística sobre esse espaço novo.

Os 17 erros que restam (Figura 3) estão na sobreposição real entre os grupos: com $\sigma = 1{,}2$ e
centros a $\lVert(3,3)\rVert \approx 4{,}24$ de distância, as caudas das gaussianas se cruzam, e
nenhum classificador acerta esses pontos. O resultado de ~92% está perto do teto do problema.

# Exercício 3 — Multiclasse, a mesma rede

## A — Gerar os dados

`make_classification` com os parâmetros do enunciado: 1500 amostras, 4 features, todas
informativas, 3 classes com 2 grupos cada, `class_sep=1.0`, `random_state=42`. Divisão 80/20
estratificada com `random_state=42`.

In [ ]:
X3, y3 = make_classification(n_samples=1500, n_features=4, n_informative=4,
                             n_redundant=0, n_repeated=0, n_classes=3,
                             n_clusters_per_class=2, class_sep=1.0,
                             random_state=42)

X3_tr, X3_te, y3_tr, y3_te = train_test_split(X3, y3, test_size=0.2,
                                              random_state=42, stratify=y3)
Y3_tr = np.eye(3)[y3_tr]           # one-hot, (N, 3)

print(f"total: {X3.shape}   treino: {X3_tr.shape}   teste: {X3_te.shape}")
print("por classe — total:", np.bincount(y3), " treino:", np.bincount(y3_tr),
      " teste:", np.bincount(y3_te))
print("desvio-padrão de cada feature (treino):", X3_tr.std(axis=0).round(3))

As classes têm 499, 499 e 502 amostras, e não 500 exatas, porque o `make_classification` tem
`flip_y=0.01` por padrão: ~1% dos rótulos é sorteado de novo ao acaso. Isso é **ruído de rótulo**
embutido nos dados, um erro que nenhum modelo elimina. A divisão estratificada deixa 100 amostras
de cada classe no teste. As quatro features já têm escala parecida (desvio-padrão entre 1,5 e
1,6 no treino), então também aqui os dados entram na rede sem padronização.

## B — Adaptar a rede

**O que muda: só a cabeça.** A camada de saída passa a ter 3 unidades, a ativação de saída vira
a softmax e a perda vira a entropia cruzada categórica. No código, isso é trocar
`CABECA_BINARIA` por `CABECA_MULTICLASSE` e o último tamanho de `[…, 1]` para `[…, 3]`. As
funções `inicializar`, `passagem_direta`, `passagem_reversa` e `atualizar` são **as mesmas**, sem
nenhuma alteração — o objeto Python é literalmente o mesmo, definido uma única vez na seção de
implementação (ponto extra).

**Derivação de $\partial L / \partial u$ para softmax + entropia cruzada.** Para uma amostra, com
pré-ativações $u_1, \dots, u_K$, probabilidades $p_k = e^{u_k} / \sum_j e^{u_j}$ e alvo one-hot
$\mathbf{y}$ ($\sum_k y_k = 1$), a perda é $L = -\sum_k y_k \log p_k$.

*1. Jacobiana da softmax.* Derivando o quociente,

$$
\frac{\partial p_k}{\partial u_j}
 = \frac{\delta_{kj}\, e^{u_k} \sum_i e^{u_i} - e^{u_k} e^{u_j}}{\big(\sum_i e^{u_i}\big)^2}
 = p_k\,(\delta_{kj} - p_j),
$$

em que $\delta_{kj} = 1$ se $k = j$ e $0$ caso contrário. Mais simples ainda é derivar o
logaritmo: $\log p_k = u_k - \log \sum_i e^{u_i}$, então
$\frac{\partial \log p_k}{\partial u_j} = \delta_{kj} - p_j$.

*2. Regra da cadeia na perda:*

$$
\frac{\partial L}{\partial u_j}
 = -\sum_k y_k \frac{\partial \log p_k}{\partial u_j}
 = -\sum_k y_k\,(\delta_{kj} - p_j)
 = -y_j + p_j \underbrace{\sum_k y_k}_{=\,1}
 = \boxed{\,p_j - y_j\,}
$$

Para a média sobre $N$ amostras, $\partial L / \partial \mathbf{U} = (\mathbf{P} - \mathbf{Y})/N$
— exatamente a função `grad_saida` usada na cabeça binária.

**Por que é tão simples.** A softmax é uma exponencial normalizada, e a entropia cruzada tira o
logaritmo dela: o $\log$ desfaz o $\exp$, e o que sobra é linear em $\mathbf{u}$ menos o termo de
normalização, cuja derivada é a própria $\mathbf{p}$. A Jacobiana cheia da softmax (uma matriz
$K \times K$) nunca precisa ser montada. O mesmo cancelamento aconteceu na cabeça binária (o
$p(1-p)$ da sigmoide cancelou o $1/[p(1-p)]$ da BCE). Não é coincidência: sigmoide + BCE é o caso
$K = 2$ de softmax + CCE, e os dois pares são a ativação "canônica" para a sua perda (a de uma
família exponencial). Há também duas consequências práticas. O sinal de erro é o próprio erro de
previsão, $\mathbf{p} - \mathbf{y}$, limitado em $[-1, 1]$; e ele nunca se anula por saturação — se a
rede prevê $p \approx 0$ para a classe certa, o gradiente é $\approx -1$, o máximo possível, ao
contrário do que acontece com o erro quadrático sobre uma saída sigmoide ou $\tanh$, em que o
fator $\sigma'(u) \approx 0$ trava o aprendizado justamente quando a rede está mais errada
(compare com o fator $1 - \hat{y}^2$ do Exercício 1).

## C — Treinar e avaliar

**Arquitetura $4 \to 16 \to 3$** ($\tanh$ na oculta, softmax na saída, entropia cruzada
categórica), 131 parâmetros, $\eta = 0{,}3$, **3000 épocas** em lote completo. As mesmas escolhas
(épocas, $\eta$) valem no Exercício 4.

**Sementes.** O `rng` global sorteia aqui cinco sementes inteiras (seu segundo e último uso). A
primeira inicializa a rede principal deste exercício e a rede principal do Exercício 4. As cinco
são usadas no estudo de sementes do item 4.B, cada uma alimentando um gerador próprio, de modo que a
rede rasa e a profunda partam da mesma semente.

In [ ]:
SEMENTES = [int(s) for s in rng.integers(0, 10_000, size=5)]     # 2º (e último) uso do rng
print("sementes dos Exercícios 3 e 4:", SEMENTES)

ARQ3, ETA34, EPOCAS34 = [4, 16, 3], 0.3, 3000

params3, perda3 = treinar_mlp(ARQ3, X3_tr, Y3_tr, CABECA_MULTICLASSE, eta=ETA34,
                              epocas=EPOCAS34, gerador=np.random.default_rng(SEMENTES[0]))

pred3 = prever(params3, X3_te, CABECA_MULTICLASSE)
acc_tr3 = (prever(params3, X3_tr, CABECA_MULTICLASSE) == y3_tr).mean()
acc_te3 = (pred3 == y3_te).mean()
print(f"\narquitetura {ARQ3}  ({contar_parametros(ARQ3)} parâmetros),  eta = {ETA34},  "
      f"{EPOCAS34} épocas,  semente {SEMENTES[0]}")
print(f"perda de treino: época 1 = {perda3[0]:.4f}   época {EPOCAS34} = {perda3[-1]:.4f}")
print(f"acurácia — treino: {acc_tr3*100:.2f}%   teste: {acc_te3*100:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(np.arange(1, EPOCAS34 + 1), perda3, lw=1.8, color="#9467bd",
        label=f"MLP {ARQ3}, $\\eta = {ETA34}$")
ax.axhline(np.log(3), ls="--", color="gray", lw=1, label=r"$\ln 3$ (prever 1/3 para cada classe)")
ax.set_xlabel("época"); ax.set_ylabel("perda de treino (entropia cruzada)")
ax.set_title("Figura 4 — Exercício 3: perda de treino × época")
ax.legend()
plt.show()

In [ ]:
cm3 = confusion_matrix(y3_te, pred3)

fig, ax = plt.subplots(figsize=(5.5, 4.8))
im = ax.imshow(cm3, cmap="Blues")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm3[i, j], ha="center", va="center", fontsize=12,
                color="white" if cm3[i, j] > cm3.max() / 2 else "black")
ax.set_xticks(range(3), [f"Classe {k}" for k in range(3)])
ax.set_yticks(range(3), [f"Classe {k}" for k in range(3)])
ax.set_xlabel("classe prevista"); ax.set_ylabel("classe verdadeira")
ax.grid(False)
fig.colorbar(im, ax=ax, label="nº de amostras de teste")
ax.set_title(f"Figura 5 — Exercício 3: matriz de confusão no teste ({acc_te3*100:.2f}%)")
plt.show()

# erros somados nos dois sentidos, por par de classes
pares = {(i, j): cm3[i, j] + cm3[j, i] for i in range(3) for j in range(i + 1, 3)}
for (i, j), n in pares.items():
    print(f"classes {i} e {j}: {cm3[i, j]} + {cm3[j, i]} = {n} confusões")
print("recall por classe:", (cm3.diagonal() / cm3.sum(axis=1)).round(3))

**Resultados.** Arquitetura $4 \to 16 \to 3$, 131 parâmetros, $\eta = 0{,}3$, 3000 épocas, semente
6779 (`SEMENTES[0]`). A perda de treino cai de 1,2479 para **0,2629** (Figura 4) e a acurácia no
teste é **83,33%** (250 de 300), contra 90,75% no treino. Com as cinco sementes do item 4.B, a
média no teste é 84,87%, dentro da faixa de 83–86% esperada.

A Figura 4 mostra uma queda quase vertical nas primeiras épocas, de $1{,}25$ para perto de 0,65 —
a parte que um modelo quase linear já consegue —, seguida de uma descida mais lenta até ~0,35 na
época 500 e de um longo trecho quase plano, ainda levemente inclinado na época 3000. Essa última
parte é a rede esculpindo as fronteiras entre grupos que se sobrepõem, e é nela que treino e
teste se separam (90,75% contra 83,33%).

## D — Análise

In [ ]:
# Linha de base linear nestes dados
base3 = LogisticRegression(max_iter=1000).fit(X3_tr, y3_tr)
acc_base3 = base3.score(X3_te, y3_te)
cm_base3 = confusion_matrix(y3_te, base3.predict(X3_te))
print(f"regressão logística multinomial — acurácia no teste: {acc_base3*100:.2f}%")
print("matriz de confusão da linha de base:\n", cm_base3)

# Estrutura de grupos: 2-means (escrito à mão) dentro de cada classe, para achar
# os 6 grupos gerados pelo make_classification. Início determinístico: os dois
# pontos extremos ao longo da direção de maior variância da classe.
def dois_grupos(Xc, iters=50):
    Xc0 = Xc - Xc.mean(axis=0)
    v = np.linalg.svd(Xc0, full_matrices=False)[2][0]
    C = np.stack([Xc[np.argmin(Xc0 @ v)], Xc[np.argmax(Xc0 @ v)]])
    for _ in range(iters):
        r = np.argmin(((Xc[:, None, :] - C[None]) ** 2).sum(axis=2), axis=1)
        C = np.stack([Xc[r == k].mean(axis=0) for k in (0, 1)])
    return C, r

centros, rotulo_grupo = [], np.empty(len(y3_tr), dtype=int)
for k in range(3):
    C, r = dois_grupos(X3_tr[y3_tr == k])
    centros += [(k, g, C[g]) for g in (0, 1)]
    rotulo_grupo[y3_tr == k] = 2 * k + r

nomes = [f"{k}{'ab'[g]}" for k, g, _ in centros]
D = np.array([[np.linalg.norm(a[2] - b[2]) for b in centros] for a in centros])
dist = pd.DataFrame(D, index=nomes, columns=nomes).round(2)
print("\ncentros dos 6 grupos (treino):")
for nome, (k, g, c) in zip(nomes, centros):
    print(f"  grupo {nome}: {np.round(c, 2)}   ({(rotulo_grupo == 2*k + g).sum()} pontos)")

# menor distância entre um grupo de uma classe e um grupo de outra, por par de classes
print("\nmenor distância entre grupos, por par de classes:")
for i in range(3):
    for j in range(i + 1, 3):
        d = min(D[2*i + a, 2*j + b] for a in (0, 1) for b in (0, 1))
        print(f"  classes {i} e {j}: {d:.2f}")
dist

**O par mais confundido.** Pela Figura 5, o par mais confundido é **Classe 0 e Classe 1**: 20
dos 50 erros do teste (9 amostras da Classe 0 previstas como 1 e 11 da Classe 1 previstas como 0).
Logo atrás vem o par **1 e 2**, com 18 (13 + 5), e bem atrás o par **0 e 2**, com 12 (7 + 5). A
diferença entre 20 e 18 é de dois pontos de teste — dentro do ruído. A conclusão robusta é outra:
**a Classe 1 é a que se confunde com as duas outras** (recall de 76%, contra 84% da Classe 0 e 90%
da Classe 2), e o par 0–2 é o que menos se confunde.

**Explicação pela estrutura de grupos.** O `make_classification` coloca os $3 \times 2 = 6$
grupos em vértices de um hipercubo em 4 dimensões, com lado $2 \cdot$ `class_sep` $= 2$, e então
aplica a cada grupo uma transformação linear aleatória (gaussianas alongadas e inclinadas, não
esféricas). Cada classe são **dois** grupos em vértices diferentes, e nada obriga os grupos da
mesma classe a estarem perto um do outro. O que decide quais classes se confundem é *quais grupos
de classes diferentes caíram em vértices vizinhos*. Para ver isso, um 2-means escrito à mão (acima)
separou cada classe nos seus dois grupos no conjunto de treino, e a tabela mostra as distâncias
entre os seis centros. (A separação é aproximada — os grupos se sobrepõem e os tamanhos encontrados
variam de 143 a 256 —, mas basta para ver a vizinhança.)

Os pares de grupos mais próximos de classes diferentes, em ordem, são 1b–2b (2,30), 0b–1a (2,56),
1b–2a (2,64), 1a–2b (2,71) e 0a–1b (2,86). **Todos envolvem a Classe 1**: cada um dos dois grupos
dela tem um vizinho da Classe 0 *e* um vizinho da Classe 2. A Classe 1 está, literalmente, no meio
das outras duas. Já o grupo mais próximo da Classe 0 para a Classe 2 está a 2,89 (0a–2a), e o
próximo par 0–2 a 3,06: são as classes mais afastadas, e é o par menos confundido. A ordem das
distâncias mínimas por par de classes (1–2: 2,30; 0–1: 2,56; 0–2: 2,89) não reproduz exatamente a
ordem 0–1 > 1–2 da matriz de confusão — com covariâncias inclinadas, a distância entre centros
não conta toda a história da sobreposição —, mas separa com clareza o par 0–2 dos outros dois.

Com `class_sep = 1` e desvios-padrão da ordem de 1 a 1,5 em cada direção, grupos a uma distância
entre 2 e 3 se sobrepõem bastante: as caudas se cruzam, e os pontos da região de sobreposição não
têm rótulo recuperável. Somado ao ~1% de rótulos sorteados ao acaso (`flip_y`), é por isso que a
acurácia satura na casa dos 80%, e é nesses pares de grupos vizinhos que os erros se concentram.

**Linha de base e o que a camada oculta acrescentou.** A regressão logística multinomial acerta
**66,33%** no teste, contra **83,33%** do MLP: **17,0 pontos percentuais** a mais (e ~18,5 p.p.
contra a média de 84,87% das cinco sementes). A matriz de confusão da linha de base mostra *onde*
está o ganho: ela confunde o par 1–2 em 57 casos (32 + 25), e o MLP em 18; já o par 0–1 vai só de
22 para 20. A regressão logística dá a cada classe uma única região **convexa** (as fronteiras
entre as classes são hiperplanos). Mas cada classe aqui são *dois* grupos em lugares diferentes,
e os grupos das Classes 1 e 2 estão **entrelaçados** (1b fica perto de 2b e de 2a; 1a fica perto
de 2b). Uma região convexa que cubra os dois grupos da Classe 1 engole também um grupo da Classe 2,
e vice-versa — é o mesmo problema do Exercício 2, em 4D e com três classes. A camada oculta deixa
a rede formar regiões **não convexas e desconexas**, uma "ilha" por grupo: ela leva os dados a um
espaço de 16 dimensões em que os seis grupos ficam aproximadamente separáveis por hiperplanos, e
a softmax faz o resto. O que a camada oculta **não** resolve é o par 0–1, cujas confusões vêm de
sobreposição real entre grupos vizinhos (0b–1a, 0a–1b), não de falta de flexibilidade da fronteira.

# Exercício 4 — Uma segunda camada oculta ajuda?

**Desenho do experimento.** Mesmos dados e mesma divisão do Exercício 3, mesma cabeça
(softmax + CCE), mesma taxa de aprendizado ($\eta = 0{,}3$), mesmo número de épocas (3000), mesmo
treino em lote completo e mesma semente. A **única** mudança é a profundidade: a rede profunda é
$4 \to 16 \to 16 \to 3$, a rasa do Exercício 3 com uma segunda camada oculta de mesma largura.
Nada foi reajustado para a rede profunda.

## A — Treinar a rede mais profunda

In [ ]:
ARQ4 = [4, 16, 16, 3]

params4, perda4 = treinar_mlp(ARQ4, X3_tr, Y3_tr, CABECA_MULTICLASSE, eta=ETA34,
                              epocas=EPOCAS34, gerador=np.random.default_rng(SEMENTES[0]))

pred4 = prever(params4, X3_te, CABECA_MULTICLASSE)
acc_tr4 = (prever(params4, X3_tr, CABECA_MULTICLASSE) == y3_tr).mean()
acc_te4 = (pred4 == y3_te).mean()
print(f"arquitetura {ARQ4}  ({contar_parametros(ARQ4)} parâmetros),  eta = {ETA34},  "
      f"{EPOCAS34} épocas,  semente {SEMENTES[0]}")
print(f"perda de treino: época 1 = {perda4[0]:.4f}   época {EPOCAS34} = {perda4[-1]:.4f}")
print(f"acurácia — treino: {acc_tr4*100:.2f}%   teste: {acc_te4*100:.2f}%")
print("matriz de confusão no teste:\n", confusion_matrix(y3_te, pred4))

**Resultado (semente 6779, a mesma da rede principal do Exercício 3).** Arquitetura
$4 \to 16 \to 16 \to 3$, **403 parâmetros** (mais de três vezes os 131 da rede rasa), $\eta = 0{,}3$,
3000 épocas: perda de treino final **0,1700** (contra 0,2629 da rasa) e acurácia no teste de
**87,33%** (treino: 93,75%), 4 pontos acima dos 83,33% da rede rasa *nesta semente*. O item B
mostra que essa vantagem é sorte da semente.

## B — Comparar

Uma semente só não basta para comparar dois modelos: a diferença entre inicializações pode ser do
mesmo tamanho que a diferença entre arquiteturas. As duas redes foram treinadas com as **cinco**
sementes sorteadas no Exercício 3, e em cada semente as duas partem do mesmo gerador. Isso
permite comparar as diferenças **pareadas** (profunda − rasa na mesma semente), além das médias.

In [ ]:
linhas = []
curvas = {"rasa": [], "profunda": []}
for s in SEMENTES:
    for nome, arq in [("rasa", ARQ3), ("profunda", ARQ4)]:
        if s == SEMENTES[0]:                         # já treinadas acima: reaproveita
            p, h = (params3, perda3) if nome == "rasa" else (params4, perda4)
        else:
            p, h = treinar_mlp(arq, X3_tr, Y3_tr, CABECA_MULTICLASSE, eta=ETA34,
                               epocas=EPOCAS34, gerador=np.random.default_rng(s))
        curvas[nome].append(h)
        linhas.append({
            "semente": s, "rede": nome,
            "perda final de treino": h[-1],
            "acurácia de treino": (prever(p, X3_tr, CABECA_MULTICLASSE) == y3_tr).mean(),
            "acurácia no teste": (prever(p, X3_te, CABECA_MULTICLASSE) == y3_te).mean(),
        })

sementes_df = pd.DataFrame(linhas)
por_semente = sementes_df.pivot(index="semente", columns="rede",
                                values=["perda final de treino", "acurácia no teste"])
por_semente = por_semente.reindex(SEMENTES)
por_semente[("diferença", "teste: profunda − rasa (p.p.)")] = 100 * (
    por_semente[("acurácia no teste", "profunda")] - por_semente[("acurácia no teste", "rasa")])
por_semente.round(4)

In [ ]:
comparacao = pd.DataFrame({
    "camadas ocultas": [1, 2],
    "arquitetura": [str(ARQ3), str(ARQ4)],
    "parâmetros": [contar_parametros(ARQ3), contar_parametros(ARQ4)],
}, index=["Exercício 3", "Exercício 4"])
for nome, idx in [("rasa", "Exercício 3"), ("profunda", "Exercício 4")]:
    g = sementes_df[sementes_df["rede"] == nome]
    comparacao.loc[idx, "perda final de treino"] = (
        f"{g['perda final de treino'].mean():.4f} ± {g['perda final de treino'].std(ddof=1):.4f}")
    comparacao.loc[idx, "acurácia de treino"] = (
        f"{g['acurácia de treino'].mean()*100:.2f}% ± {g['acurácia de treino'].std(ddof=1)*100:.2f}")
    comparacao.loc[idx, "acurácia no teste"] = (
        f"{g['acurácia no teste'].mean()*100:.2f}% ± {g['acurácia no teste'].std(ddof=1)*100:.2f}")

dif = por_semente[("diferença", "teste: profunda − rasa (p.p.)")]
print(f"média de {len(SEMENTES)} sementes (± desvio-padrão entre sementes)")
print(f"diferença pareada no teste (profunda − rasa): média {dif.mean():+.2f} p.p., "
      f"desvio {dif.std(ddof=1):.2f} p.p., profunda melhor em {(dif > 0).sum()} de {len(dif)} sementes")
comparacao

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ep = np.arange(1, EPOCAS34 + 1)
for nome, arq, cor in [("rasa", ARQ3, "#1f77b4"), ("profunda", ARQ4, "#d62728")]:
    H = np.array(curvas[nome])
    for h in H[1:]:
        ax.plot(ep, h, color=cor, lw=0.7, alpha=0.35)
    ax.plot(ep, H[0], color=cor, lw=2.2,
            label=f"Ex. {3 if nome == 'rasa' else 4}: {arq}, semente {SEMENTES[0]}")
ax.plot([], [], color="gray", lw=0.7, alpha=0.6, label="outras 4 sementes (linhas finas)")
ax.set_yscale("log")
ax.set_xlabel("época"); ax.set_ylabel("perda de treino (entropia cruzada, escala log)")
ax.set_title(f"Figura 6 — Exercícios 3 e 4: perda de treino, 1 vs. 2 camadas ocultas "
             f"($\\eta = {ETA34}$)")
ax.legend()
plt.show()

**Tabela de comparação** (média de 5 sementes; ± é o desvio-padrão entre sementes):

| | camadas ocultas | parâmetros | perda final de treino | acurácia no teste |
|:---|:---:|:---:|:---:|:---:|
| Exercício 3 | 1 ($4 \to 16 \to 3$) | 131 | $0{,}2601 \pm 0{,}0054$ | $84{,}87\% \pm 0{,}99$ |
| Exercício 4 | 2 ($4 \to 16 \to 16 \to 3$) | 403 | $0{,}1709 \pm 0{,}0114$ | $85{,}07\% \pm 1{,}69$ |

**Diferença no teste (profunda − rasa), pareada por semente:** $+4{,}00$, $-2{,}67$, $-1{,}33$,
$+1{,}67$ e $-0{,}67$ p.p. — média **$+0{,}20$ p.p.**, desvio-padrão 2,64 p.p. A rede profunda
ganha em 2 das 5 sementes e perde em 3. O erro-padrão da média é $2{,}64/\sqrt{5} \approx 1{,}2$
p.p., seis vezes maior que a diferença: **as duas redes são estatisticamente indistinguíveis no
teste.** No treino a história é outra: a perda final da profunda é menor em *todas* as cinco
sementes — de 25% a 39% menor, 34% na média (0,171 contra 0,260) — e a acurácia de treino é 3,6 p.p.
maior (93,90% contra 90,33%).

Note a primeira semente. Sozinha, ela diria que a profundidade compra 4 pontos — exatamente o
tipo de conclusão que o enunciado pede para evitar ao exigir várias sementes.

**Figura 6.** As curvas grossas são a semente principal; as finas, as outras quatro. Até a época
~100 as duas redes descem juntas. A partir daí a profunda fica sempre abaixo, e a distância
**cresce**: a curva rasa está quase plana na época 3000 (já perto do que 16 unidades conseguem),
enquanto a profunda continua caindo em linha reta na escala log. A profunda também tem **picos**
— saltos bruscos da perda em algumas sementes (perto das épocas 630, 1760 e 2710), seguidos de
recuperação — que não aparecem na rasa: com o mesmo $\eta = 0{,}3$, a rede de duas camadas fica
perto do limite de estabilidade do gradiente descendente.

## C — Análise

**O que foi medido.** A segunda camada oculta não melhorou a acurácia no teste
($+0{,}20 \pm 2{,}64$ p.p., indistinguível de zero), mas reduziu a perda de treino em um terço.
Ou seja, a profundidade comprou **ajuste ao treino**, não **generalização**. A brecha entre treino
e teste cresceu de 5,5 p.p. (90,33% − 84,87%) para 8,8 p.p. (93,90% − 85,07%).

**Por que a profundidade não ajuda aqui — a estrutura dos dados.** O problema são seis gaussianas
em 4D. A fronteira ótima entre duas gaussianas é uma superfície quadrática, e a fronteira entre
seis grupos é uma colagem de poucas superfícies suaves: não há estrutura hierárquica, não há
padrão que se repita em várias escalas, não há "partes" que se combinem em "objetos". Uma camada de
16 unidades $\tanh$ já tem flexibilidade de sobra para desenhar ilhas em volta de seis grupos (o
Exercício 2 precisou, em princípio, de duas unidades por "faixa"). O que limita a acurácia em ~85%
**não é falta de expressividade, é sobreposição**: os grupos vizinhos estão a distâncias de 2 a 3
com desvios de 1 a 1,5, e 1% dos rótulos são aleatórios. Nenhuma fronteira, rasa ou profunda,
recupera o rótulo de um ponto que está no meio da sobreposição.

**Representação contra otimização.** O que a segunda camada acrescenta é *composição*: a camada 2
combina as 16 "retas suaves" da camada 1 em regiões, e a saída combina regiões. Isso é poder
expressivo que estes dados não pedem. O custo, por outro lado, aparece:

- **mais parâmetros** (403 contra 131) para 1200 amostras de treino — capacidade que, sem um sinal
  real a capturar, vai para os detalhes da amostra de treino (os pontos da região de sobreposição e
  os rótulos sorteados);
- **otimização mais difícil**: o gradiente atravessa duas $\tanh$ e duas matrizes de pesos, o que
  multiplica fatores e deixa o passo efetivo maior; com o $\eta$ fixado para a rede rasa, a
  profunda tem os picos de instabilidade da Figura 6;
- **mais variância entre sementes**: desvio de 1,69 p.p. no teste contra 0,99 da rasa, e desvio da
  perda final duas vezes maior. Uma superfície de perda com mais parâmetros tem mais mínimos
  diferentes, e cada semente cai num.

**A perda de treino conta a mesma história que a acurácia no teste? Não.** Pela Figura 6, a rede
profunda é claramente "melhor" — perda 34% menor na média, menor em todas as sementes, e ainda
caindo. Pela
acurácia no teste, é igual. A queda extra da perda vem de deixar as probabilidades mais confiantes
nos pontos de treino e de acertar pontos de treino que estão do lado "errado" da sobreposição —
coisas que não se transferem para pontos novos. E a curva profunda ainda descendo na época 3000
indica que, com mais épocas, a brecha cresceria. A perda de treino mede o quanto o modelo se ajusta
aos dados que viu; só um conjunto separado mede o que ele aprendeu sobre o problema.

**Que dados recompensariam a profundidade extra.** Dados em que a função alvo é uma **composição**
de funções mais simples, com estrutura que se reaproveita:

- fronteiras com **muitas** regiões que se repetem — um tabuleiro de xadrez com dezenas de casas,
  uma espiral com várias voltas. O exemplo teórico clássico é o "dente de serra" obtido compondo
  $k$ vezes uma função em tenda: ele tem $2^k$ oscilações, uma rede com $k$ camadas de poucas
  unidades o representa exatamente, e uma rede rasa precisa de largura exponencial em $k$ para
  aproximá-lo (Telgarsky, 2016). Cada camada nova *dobra* o número de regiões; numa camada só, cada
  região custa unidades próprias;
- dados **hierárquicos**, como imagens (bordas → texturas → partes → objetos), áudio e texto, em que
  as características úteis de um nível são construídas a partir das do nível de baixo;
- e, em qualquer caso, **muito mais dados** do que os 1200 pontos aqui, para que a capacidade extra
  seja usada para capturar estrutura e não ruído.

Neste problema, com seis grupos gaussianos sobrepostos em 4D, a profundidade não tem o que
compor — e o resultado honesto é que ela não ajuda.

# Resumo dos resultados

| # | Quantidade | Valor |
|:-:|:---|:---|
| 1 | Ex. 1 — $\mathbf{z}^{(1)}$ e $\mathbf{h}^{(1)}$ | $\mathbf{z}^{(1)} = [0{,}270000,\ -0{,}180000]$, $\ \mathbf{h}^{(1)} = [0{,}263625,\ -0{,}178081]$ |
| 2 | Ex. 1 — $u^{(2)}$, $\hat{y}$ e $L$ | $u^{(2)} = 0{,}385237$, $\ \hat{y} = 0{,}367247$, $\ L = 0{,}400377$ |
| 3 | Ex. 1 — os oito parâmetros atualizados | $\mathbf{W}^{(2)} = [0{,}586587,\ -0{,}358490]$, $b^{(2)} = 0{,}528448$, $\mathbf{W}^{(1)} = \begin{bmatrix} 0{,}376405 & -0{,}130562 \\ 0{,}152295 & 0{,}419082 \end{bmatrix}$, $\mathbf{b}^{(1)} = [0{,}252811,\ -0{,}295410]$; a perda cai para $0{,}083417$ |
| 4 | Ex. 2 — acurácia da linha de base linear | **47,50%** (regressão logística) |
| 5 | Ex. 2 — arquitetura e acurácia no teste | $2 \to 8 \to 1$ ($\tanh$/sigmoide, BCE, $\eta = 0{,}3$, 2000 épocas): **91,50%** |
| 6 | Ex. 3 — arquitetura e acurácia no teste | $4 \to 16 \to 3$ ($\tanh$/softmax, CCE, $\eta = 0{,}3$, 3000 épocas): **83,33%** (semente 6779); **84,87%** na média de 5 sementes |
| 7 | Ex. 3 — par de classes mais confundido | **Classes 0 e 1**: 20 dos 50 erros (9 + 11); o par 1–2 vem logo atrás com 18 e o par 0–2 tem 12 |
| 8 | Ex. 4 — arquitetura e acurácia no teste | $4 \to 16 \to 16 \to 3$ (mesmos $\eta$, épocas e sementes): **87,33%** (semente 6779); **85,07%** na média de 5 sementes |
| 9 | Ex. 4 — diferença de acurácia em relação ao Ex. 3 (média de ≥ 3 sementes) | **$+0{,}20$ p.p.** (média pareada de 5 sementes; desvio 2,64 p.p.; profunda melhor em 2 de 5) — sem diferença real |

Conferência gerada pelo código, a partir das mesmas variáveis usadas acima:

In [ ]:
par_mais = max(pares, key=pares.get)
media = lambda nome, col: sementes_df.loc[sementes_df["rede"] == nome, col].mean()

linhas_resumo = [
    (1, "Ex. 1 — z1 e h1", f"z1 = {np.round(z1, 6)},  h1 = {np.round(h1, 6)}"),
    (2, "Ex. 1 — u2, y_hat e L", f"u2 = {u2:.6f},  y_hat = {y_hat:.6f},  L = {L:.6f}"),
    (3, "Ex. 1 — parâmetros atualizados",
        f"W2 = {np.round(W2n, 6)}, b2 = {b2n:.6f}, W1 = {np.round(W1n, 6).tolist()}, "
        f"b1 = {np.round(b1n, 6)};  L na 2ª passagem = {L_n:.6f}"),
    (4, "Ex. 2 — acurácia da linha de base linear", f"{acc_base2*100:.2f}%"),
    (5, "Ex. 2 — arquitetura e acurácia no teste",
        f"{ARQ2}, eta = {ETA2}, {EPOCAS2} épocas: {acc_te2*100:.2f}%"),
    (6, "Ex. 3 — arquitetura e acurácia no teste",
        f"{ARQ3}, eta = {ETA34}, {EPOCAS34} épocas: {acc_te3*100:.2f}% (semente {SEMENTES[0]}); "
        f"média de {len(SEMENTES)} sementes {media('rasa', 'acurácia no teste')*100:.2f}%"),
    (7, "Ex. 3 — par de classes mais confundido",
        f"classes {par_mais[0]} e {par_mais[1]}: {pares[par_mais]} de {cm3.sum() - np.trace(cm3)} erros"),
    (8, "Ex. 4 — arquitetura e acurácia no teste",
        f"{ARQ4}: {acc_te4*100:.2f}% (semente {SEMENTES[0]}); "
        f"média de {len(SEMENTES)} sementes {media('profunda', 'acurácia no teste')*100:.2f}%"),
    (9, "Ex. 4 — diferença em relação ao Ex. 3 (média de sementes)",
        f"{dif.mean():+.2f} p.p. (desvio entre sementes {dif.std(ddof=1):.2f} p.p.)"),
]
resumo = pd.DataFrame(linhas_resumo, columns=["#", "Quantidade", "Valor"]).set_index("#")
with pd.option_context("display.max_colwidth", 250):
    display(resumo.style.set_properties(**{"text-align": "left"}))

### Leitura dos números

**Exercício 1.** Um passo de retropropagação com $\eta = 0{,}3$ leva $\hat{y}$ de 0,367 para 0,711 e
a perda de 0,400 para 0,083. Todos os nove números mudam no sentido de aumentar $u^{(2)}$, porque a
saída estava abaixo do alvo, e cada gradiente é o produto do erro que chega com a derivada local.

**Exercício 2.** Os dados têm a Classe 1 dos dois lados da Classe 0, e as duas classes têm o mesmo
centro. A regressão logística fica sem direção para escolher e acerta 47,50%, no nível do acaso. Um
MLP com uma camada oculta de 8 unidades desenha a faixa diagonal que nenhuma reta desenha e chega
a 91,50%, perto do limite imposto pela sobreposição dos grupos.

**Exercício 3.** A mesma rede, trocando só a cabeça, resolve o problema de três classes: 83,33%
(84,87% na média de sementes) contra 66,33% da linha de base linear. O ganho está quase todo no
par 1–2, cujos grupos estão entrelaçados; os erros que restam vêm de grupos vizinhos que se
sobrepõem, sobretudo os da Classe 1, que fica entre as outras duas.

**Exercício 4.** Uma segunda camada oculta reduz a perda de treino em um terço, mas não muda a
acurácia no teste ($+0{,}20$ p.p. em média, com desvio de 2,64 p.p. entre sementes). Os dados não
têm estrutura composicional para a profundidade explorar, e o limite aqui é a sobreposição, não a
expressividade. Uma única semente teria sugerido um ganho de 4 pontos que não existe.